# NIFTY 100 Time-Series Modelling: ARMA, GARCH, VAR

This companion notebook implements the workflow described in `PROJECT_REPORT.md` using the full enriched NIFTY 100 dataset in `testing/NIFTY_100_enriched.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# Ensure plots display inline if run in Jupyter
%matplotlib inline

nifty_path = Path('..') / 'testing' / 'NIFTY_100_enriched.csv'

df = pd.read_csv(nifty_path, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

df['log_return'] = np.log(df['close']).diff()
df = df.dropna().reset_index(drop=True)

df.head()

## Exploratory analysis

We inspect the price series and log returns to confirm non-stationarity in levels and approximate stationarity in returns, as argued in the report.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(df['date'], df['close'])
axes[0].set_title('NIFTY 100 Close')
axes[0].set_ylabel('Index level')

axes[1].plot(df['date'], df['log_return'])
axes[1].set_title('NIFTY 100 Log Returns')
axes[1].set_ylabel('Log return')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

## ARMA(1,1) model on log returns

We fit an ARMA(1,1) model using the `ARMAModel` class from `src/timeseries.py`, then inspect residuals and their autocorrelation.


In [ ]:
import sys
from pathlib import Path

# Add src to path
src_path = Path('..') / 'src'
sys.path.append(str(src_path))

from timeseries import ARMAModel, autocorrelation

returns = df['log_return'].values

arma = ARMAModel(p=1, q=1)
arma.fit(returns)

resid = arma.residuals

plt.figure(figsize=(10, 3))
plt.plot(resid)
plt.title('ARMA(1,1) residuals')
plt.xlabel('Time')
plt.ylabel('Residual')
plt.tight_layout()
plt.show()

# Autocorrelation of residuals
lags = 30
acf_resid = autocorrelation(resid, max_lag=lags)

plt.figure(figsize=(10, 3))
plt.stem(range(1, lags + 1), acf_resid[1:], use_line_collection=True)
plt.title('Residual autocorrelation')
plt.xlabel('Lag')
plt.ylabel('ACF')
plt.tight_layout()
plt.show()

## GARCH(1,1) model for volatility

We fit a GARCH(1,1) model to the ARMA residuals using `GARCHModel` and visualise the conditional variance series.


In [ ]:
from timeseries import GARCHModel

# Mean-removed residuals
eps = resid - np.mean(resid)

garch = GARCHModel(p=1, q=1)
garch.fit(eps)

sigma2 = garch.conditional_variances

plt.figure(figsize=(10, 3))
plt.plot(sigma2)
plt.title('GARCH(1,1) conditional variance')
plt.xlabel('Time')
plt.ylabel('Variance')
plt.tight_layout()
plt.show()

## Simple VAR(1) example

For illustration, we construct a two-series dataset from log returns and a rolling volatility proxy, then fit a VAR(1) model.


In [ ]:
from timeseries import VARModel

window = 20
roll_var = df['log_return'].rolling(window).var()
roll_var = roll_var.shift(1)  # lag to avoid look-ahead

var_df = pd.DataFrame({
    'returns': df['log_return'],
    'roll_var': roll_var
}).dropna().reset_index(drop=True)

Y = var_df.values

var_model = VARModel(p=1)
var_model.fit(Y)

# Simple one-step forecast from the last observed vector
last = Y[-1:]
forecast = var_model.forecast(last, steps=1)
forecast

This notebook provides figures and numeric outputs that correspond directly to the workflow and qualitative findings described in `PROJECT_REPORT.md`, and can be used to generate tables and confidence intervals for the final submitted report.
